# Step 2 — Compare conditions: normalisation, statistics and plots

This notebook reads the per-cell `per_z.csv` files made by step 1, normalises each cell to the control cells, tests every other condition against the control and makes the plots. One or more datasets can be analysed side by side, for example live and fixed cells.

**Per cell** (from the background-subtracted `per_z.csv`):
- mean intensity: the mean over all z slices of the mean inside the mask in each slice (slices with no mask pixels are skipped)
- pixel number: the total number of mask pixels over all z slices

**Normalisation:** each value is divided by the mean of all control cells of the same dataset, pooled over the experiments.

**Statistics:** Shapiro–Wilk test of normality per group, then a two-sided Mann–Whitney U test of each condition against the control. Cells are the statistical unit, and there is no correction for multiple comparisons when there are more than two conditions.

Set the datasets and conditions in section 2, then run all cells from top to bottom. Tables and plots are written to `OUTPUT_DIR`.

## 1 — Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

## 2 — Settings

In [ ]:
# One entry per dataset: the OUTPUT_DIR of step 1, and condition folder -> short label
DATASETS = {
    'dataset_1': {'folder': Path('data/3_Processed'),
                  'conditions': {'control_folder': 'Ctrl', 'treated_folder': 'Treated'}},
}

CONTROL = 'Ctrl'                                     # label of the control condition
ORDER   = ['Ctrl', 'Treated']                        # plotting order
LABELS  = {'Ctrl': 'Control', 'Treated': 'Treated'}  # names shown on the plots
PALETTE = {'Ctrl': 'thistle', 'Treated': 'deeppink'}

Z_YLIM = None   # y limits of the plots along z, e.g. (0, 12); None = automatic

OUTPUT_DIR = Path('output')

assert CONTROL in ORDER, 'CONTROL must be in ORDER'
for name, dataset in DATASETS.items():
    assert dataset['folder'].is_dir(), f'{name}: folder not found: {dataset["folder"].resolve()}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3 — Load the per-z tables

In [ ]:
def load_per_z(folder, conditions):
    # One row per cell and z slice, with condition and experiment from the folder names
    tables = []
    for cond_folder, condition in conditions.items():
        for path in sorted((folder / cond_folder).glob('*/*/per_z.csv')):
            df = pd.read_csv(path)
            df['condition'] = condition
            df['experiment'] = path.parent.parent.name.rsplit('_Exp', 1)[-1]
            tables.append(df)
    assert tables, f'No per_z.csv files found in {folder}'
    return pd.concat(tables, ignore_index=True)


per_z = {name: load_per_z(d['folder'], d['conditions']) for name, d in DATASETS.items()}

for name, df in per_z.items():
    counts = df.drop_duplicates(['cell', 'condition', 'experiment']).groupby(['condition', 'experiment']).size().unstack(fill_value=0)
    counts['total'] = counts.sum(axis=1)
    print(f'\n{name} cells per experiment:'); print(counts.to_string())

## 4 — Per-cell values and normalisation

In [ ]:
def channel_columns(df):
    return [c for c in df.columns if c.endswith('_mean')]


def per_cell(df):
    # Mean over z of each channel's mean inside the mask, and total mask pixels, per cell
    columns = channel_columns(df)
    cell = df.groupby(['cell', 'condition', 'experiment']).agg(
        **{c: (c, 'mean') for c in columns}, total_pixels=('n_pixels', 'sum')
    ).reset_index()
    control = cell[cell['condition'] == CONTROL]
    for col in columns + ['total_pixels']:
        cell[col + '_normalized'] = cell[col] / control[col].mean()
    return cell


cells = {name: per_cell(df) for name, df in per_z.items()}

summary = pd.concat([c.assign(dataset=name) for name, c in cells.items()], ignore_index=True)
summary.to_csv(OUTPUT_DIR / 'per_cell_summary.csv', index=False)
print(f'Saved per_cell_summary.csv ({len(summary)} cells)')

## 5 — Statistics (each condition vs control)

In [ ]:
def shapiro_p(values):
    return stats.shapiro(values).pvalue if len(values) >= 3 else np.nan


rows = []
for name, c in cells.items():
    for col in channel_columns(per_z[name]) + ['total_pixels']:
        ctrl = c.loc[c['condition'] == CONTROL, col + '_normalized'].dropna()
        for condition in [o for o in ORDER if o != CONTROL]:
            other = c.loc[c['condition'] == condition, col + '_normalized'].dropna()
            if ctrl.empty or other.empty:
                continue
            u = stats.mannwhitneyu(other, ctrl, alternative='two-sided')
            rows.append({'dataset': name, 'measure': col, 'condition': condition, 'control': CONTROL,
                         'n_control': len(ctrl), 'n_condition': len(other),
                         'mean_control': ctrl.mean(), 'mean_condition': other.mean(),
                         'shapiro_p_control': shapiro_p(ctrl), 'shapiro_p_condition': shapiro_p(other),
                         'mannwhitney_U': u.statistic, 'mannwhitney_p': u.pvalue})

stats_table = pd.DataFrame(rows)
stats_table.to_csv(OUTPUT_DIR / 'statistics.csv', index=False)
pd.set_option('display.float_format', '{:.4g}'.format)
stats_table

## 6 — Normalised values per cell

In [ ]:
def violin_panel(ax, data, col, title):
    sns.violinplot(data=data, x='condition', y=col, order=ORDER, hue='condition', hue_order=ORDER,
                   palette=PALETTE, inner=None, linewidth=2, alpha=0.5, legend=False, ax=ax)
    sns.boxplot(data=data, x='condition', y=col, order=ORDER, width=0.3, fliersize=0, linewidth=1.5,
                boxprops={'facecolor': 'none'}, ax=ax)
    sns.swarmplot(data=data, x='condition', y=col, order=ORDER, hue='condition', hue_order=ORDER,
                  palette=PALETTE, edgecolor='grey', linewidth=0.5, size=4, legend=False, ax=ax)
    n = data['condition'].value_counts()
    ax.set_xticks(range(len(ORDER)), [f'{LABELS[t]}\n(n={n.get(t, 0)})' for t in ORDER])
    ax.set_xlabel('')
    ax.set_title(title)


for name, c in cells.items():
    for col in channel_columns(per_z[name]) + ['total_pixels']:
        fig, ax = plt.subplots(figsize=(3.2, 5))
        violin_panel(ax, c, col + '_normalized', name)
        ax.set_ylabel('Normalized pixel number' if col == 'total_pixels' else f'Normalized mean {col[:-len("_mean")]} intensity')
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / f'{name}_{col}_normalized.svg')
        plt.show()

## 7 — Intensity along z (mean ± SEM per z slice; z = 0 is the first slice)

In [ ]:
for name, df in per_z.items():
    for col in channel_columns(df):
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.lineplot(data=df, x='z', y=col, hue='condition', hue_order=ORDER, palette=PALETTE, errorbar='se', ax=ax)
        ax.set_xlabel('z slice')
        ax.set_ylabel(f'Mean {col[:-len("_mean")]} intensity (a.u.)')
        if Z_YLIM:
            ax.set_ylim(*Z_YLIM)
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles, [LABELS[t] for t in labels], title='')
        ax.set_title(name)
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / f'{name}_{col}_along_z.svg')
        plt.show()